# 03 - Let's estimate the uplift with DML

**Business Loves Uplift: Causal Impact Evaluation With Python**<br>
ODSC AI West 2026 · Alex Molak · [CausalPython.io](https://causalpython.io)

<a href="https://colab.research.google.com/github/AlxndrMlk/talks-and-workshops/blob/main/2026-10--odsc-ai/03%20-%20Double%20machine%20learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>


**The story.** A retailer's field sales reps each look after 120-200 stores. The data team built a recommender that tells a rep which new products (SKUs) to list in each store. How much does it lift store sales?

The data is synthetic, so we know the true answer. Weekly sales are in k$ per store.

**The rollout.** After the A/B test, the recommender went live for every rep in the North (week 76). Management picked the North because "the North team is our strongest". Did it work, and how much?

We have one row per rep. The North reps are treated, the other 210 reps aren't. The reps weren't randomized, so we need a causal graph, an adjustment set, and an estimator that doesn't overfit: double machine learning (DML).

In [ ]:
# Colab only: install the packages and get the data generator (locally, use the workshop env)
import sys
if "google.colab" in sys.modules:
    !pip install -q dowhy==0.14 econml==0.17.0 cluster-experiments==0.31.0 mlsynth==1.0.0
    !wget -q -nc https://raw.githubusercontent.com/AlxndrMlk/talks-and-workshops/main/2026-10--odsc-ai/uplift_data.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import statsmodels.formula.api as smf

from dowhy import CausalModel
from econml.dml import LinearDML
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import uplift_data as ud

BLUE, GOLD, GREY, RED, DARK = "#0057d9", "#c8860d", "#bbbbbb", "#c0392b", "#222222"
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
d = ud.make_datasets()     # seed 23: everyone gets the same world
truth = d["truth"]         # the answer key; we only peek at it at the end of each section
print(ud.fingerprint(d))   # should print 13302.553/5496.520

## The data

| column | meaning |
|---|---|
| `north` | 1 for North reps (treatment) |
| `sales_post` | mean weekly sales per store after the rollout, weeks 76-95 (outcome) |
| `experience` | years in the job |
| `n_stores`, `store_size`, `footfall`, `urban_share` | the rep's portfolio |
| `sales_pre` | mean weekly sales per store before the rollout, weeks 24-75 |
| `new_skus` | new products listed per store per month, after the rollout |
| `bonus` | 1 if the rep got a management bonus this year |

In [ ]:
reps = d["reps_observed"]
reps.head()

In [ ]:
reps.groupby("north")[["experience", "store_size", "footfall", "urban_share", "sales_pre", "sales_post"]].mean().round(2)

North reps are more experienced and their stores sold more **before** the rollout. A plain difference in post-period sales mixes the effect with these differences.

In [ ]:
naive = reps.groupby("north").sales_post.mean().diff().iloc[-1]
print(f"naive difference: {naive:.2f} k$ per store-week")

## The graph

- **Confounders** affect both who got the recommender (North) and sales: experience and the portfolio.
- **Mediator:** the recommender works by getting new SKUs listed. `new_skus` sits on the causal path.
- **Collider:** management handed out bonuses to the pilot reps and to reps whose sales grew. `bonus` is caused by both.
- **Hidden confounder:** what if something we didn't measure drives both the choice of the North and sales growth? We draw it as a dashed node. It's not in the data.

In [ ]:
W = ["store_size", "footfall", "urban_share", "sales_pre", "n_stores"]   # portfolio
confounders = ["experience"] + W

G = nx.DiGraph()
for c in confounders:
    G.add_edges_from([(c, "north"), (c, "sales_post")])
G.add_edges_from([
    ("north", "new_skus"), ("new_skus", "sales_post"),   # mediator
    ("north", "bonus"), ("sales_post", "bonus"),         # collider
])

# draw it, with the hidden confounder U added for the picture only
xs = [-4.2, -3.0, -1.8, 1.8, 3.0, 4.2]   # a gap in the middle for U
pos = {c: (x, 2.2) for x, c in zip(xs, confounders)}
pos.update({"north": (-2.2, 0), "sales_post": (2.2, 0), "new_skus": (0, -0.9), "bonus": (0, -1.9), "U?": (0, 2.2)})
colors = {"north": BLUE, "sales_post": BLUE, "new_skus": GOLD, "bonus": RED}

plt.figure(figsize=(11, 5.5))
nx.draw_networkx_edges(G, pos, edge_color=GREY, arrowsize=14, node_size=2600)
nx.draw_networkx_edges(nx.DiGraph([("U?", "north"), ("U?", "sales_post")]), pos, style="dashed",
                       edge_color=RED, arrowsize=14, node_size=2600)
nx.draw_networkx_nodes(G, pos, node_size=2600, node_color=[colors.get(n, "#e9e4d4") for n in G.nodes])
nx.draw_networkx_nodes(nx.DiGraph([("U?", "U?")]), pos, nodelist=["U?"], node_size=2600, node_color="white", edgecolors=RED)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_labels(G, pos, labels={"U?": "U?"}, font_size=9)
plt.axis("off")
plt.show()

## Identification with DoWhy

We give DoWhy the graph without `U`: our working assumption is that there's no hidden confounder. We'll stress test that assumption later.

DoWhy reads the adjustment set off the graph. Note that it leaves out the mediator and the collider.

In [ ]:
model = CausalModel(data=reps.drop(columns=["rep_id", "region"]),
                    treatment="north", outcome="sales_post", graph=G)
estimand = model.identify_effect()
print("adjust for:", estimand.get_backdoor_variables())

## Estimation with DML

DML fits two ML models: one predicts sales from the confounders, the other predicts the treatment. It then regresses the sales residuals on the treatment residuals, with cross-fitting so no row is predicted by a model that saw it.

We hand DoWhy's adjustment set to `LinearDML` from EconML. `experience` enters as `X`: the effect may change with it (the CATE). The rest of the adjustment set enters as `W`. We ask for the **ATT**: the average effect for the North reps, the ones who got the recommender.

In [ ]:
adjust = estimand.get_backdoor_variables()
W = [c for c in W if c in adjust]   # the portfolio part of the set; experience goes in X


def linear_dml(df, w_cols, y="sales_post"):
    """LinearDML with X = experience; returns the fitted model and the ATT for North reps."""
    est = LinearDML(model_y=make_pipeline(StandardScaler(), LassoCV()),
                    model_t=make_pipeline(StandardScaler(), LogisticRegressionCV(max_iter=2000)),
                    discrete_treatment=True, cv=5, random_state=0)
    est.fit(df[y], df.north, X=df[["experience"]], W=df[w_cols])
    x_north = df.loc[df.north == 1, ["experience"]]
    lo, hi = est.ate_interval(X=x_north)
    return est, est.ate(X=x_north), lo, hi


dml, att, lo, hi = linear_dml(reps, W)
north_post = reps.loc[reps.north == 1, "sales_post"].mean()
print(f"DML ATT: {att:.2f} k$ [{lo:.2f}, {hi:.2f}] = {att / (north_post - att):.1%} of what the North would sell without it")

## The CATE: who benefits most?

The effect changes with experience. Juniors get more out of the recommender.

In [ ]:
grid = pd.DataFrame({"experience": np.linspace(reps.experience.min(), reps.experience.max(), 50)})
cate = dml.effect(grid)
cate_lo, cate_hi = dml.effect_interval(grid)

plt.fill_between(grid.experience, cate_lo, cate_hi, color=BLUE, alpha=0.15)
plt.plot(grid.experience, cate, color=BLUE, label="DML: CATE(experience)")
plt.axhline(0, color=GREY, ls="--")
plt.xlabel("rep experience (years)")
plt.ylabel("effect, k$ per store-week")
plt.legend(frameon=False)
plt.show()
print(f"slope: {dml.coef_[0]:.2f} k$ per extra year of experience")

## Bad controls

More features isn't better identification. What if we "control for everything", including the mediator or the collider?

This is an identification problem, not an estimation one, so any estimator shows it. We use a plain linear regression here: it's fast and its intervals behave. (With the mediator, DML's interval explodes: `new_skus` nearly tells who's in the North, so little treatment variation is left. Try it.)

In [ ]:
base_formula = "sales_post ~ north + " + " + ".join(adjust)
rows = []
for label, extra in [("graph's adjustment set", ""),
                     ("+ new_skus (mediator)", " + new_skus"),
                     ("+ bonus (collider)", " + bonus")]:
    fit = smf.ols(base_formula + extra, reps).fit()
    lo_, hi_ = fit.conf_int().loc["north"]
    rows.append({"controls": label, "effect": fit.params["north"], "lo": lo_, "hi": hi_})
bad = pd.DataFrame(rows)
bad.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.errorbar(bad.effect, bad.controls, xerr=[bad.effect - bad.lo, bad.hi - bad.effect], fmt="o", color=BLUE, capsize=4)
ax.axvline(0, color=GREY, ls="--")
ax.invert_yaxis()
ax.set_xlabel("effect of north, k$ per store-week (95% CI)")
plt.show()

Controlling for the mediator blocks the path the effect travels on, so most of the effect disappears. Controlling for the collider opens a path between the treatment and the outcome that isn't causal, and pulls the estimate down. DML can't fix a wrong adjustment set: identification comes from the graph.

## What if there's a confounder we didn't measure?

DoWhy's sensitivity analysis asks: how strong would a hidden confounder have to be to change our conclusion? Each point on the plot is a possible confounder, described by how much of the treatment (x) and of the outcome (y) it explains, as partial R². The contours show the estimate after removing its bias.

We can't measure the hidden confounder, but we can compare it to one we know: the triangles are confounders 1x, 2x and 3x as strong as rep experience.

The partial R² method needs a linear model, so we run it on DoWhy's linear regression estimate. That estimate is close to the DML one.

In [ ]:
linear = model.estimate_effect(estimand, method_name="backdoor.linear_regression")
print(f"linear regression estimate: {linear.value:.2f} k$ (DML ATT: {att:.2f} k$)")

sensitivity = model.refute_estimate(
    estimand, linear,
    method_name="add_unobserved_common_cause",
    simulation_method="linear-partial-R2",
    benchmark_common_causes=["experience"],
    effect_fraction_on_treatment=[1, 2, 3],
    plot_estimate=False,
)
print(sensitivity)

In [ ]:
sensitivity.plot(plot_type="estimate")

In [ ]:
benchmarks = sensitivity.benchmarking_results
benchmarks.insert(0, "as strong as experience", ["1x", "2x", "3x"])
benchmarks.round(3)

A confounder up to 3x as strong as experience would shrink the estimate, but not wipe it out: the red line (effect = 0) is far away. Our conclusion "the recommender works" survives. The exact number is less certain: depending on how strong the hidden confounder is, the effect could be anywhere in the bias-adjusted range above.

## Answer key

The generator has the hidden confounder: **relationship strength** between the rep and their store managers. Strong relationships make sales grow faster, and the North team really is the strongest. `reps_full` has the column.

In [ ]:
full = d["reps_full"]
_, att_full, lo_f, hi_f = linear_dml(full, W + ["relationship_strength"])
att_obs = att
print(f"true ATT (North reps):           {truth['att_north_rep_abs']:.2f} k$")
print(f"DML, observed data:              {att_obs:.2f} k$")
print(f"DML, with relationship strength: {att_full:.2f} k$ [{lo_f:.2f}, {hi_f:.2f}]")

In [ ]:
# where does the real hidden confounder sit on the sensitivity plot?
rest = " + ".join(confounders)
m_y = smf.ols(f"sales_post ~ north + relationship_strength + {rest}", full).fit()
m_t = smf.ols(f"north ~ relationship_strength + {rest}", full).fit()
partial_r2 = lambda m: m.tvalues["relationship_strength"] ** 2 / (m.tvalues["relationship_strength"] ** 2 + m.df_resid)
u_t, u_y = partial_r2(m_t), partial_r2(m_y)

plt.figure(figsize=(5, 4))
plt.scatter(benchmarks.r2tu_w, benchmarks.r2yu_tw, marker="^", color=RED, s=60, label="1x, 2x, 3x experience")
for _, b in benchmarks.iterrows():
    plt.annotate(b["as strong as experience"], (b.r2tu_w, b.r2yu_tw), textcoords="offset points", xytext=(6, -3))
plt.scatter([u_t], [u_y], color=BLUE, s=80, label="relationship strength (the real U)")
plt.xlabel("partial R² with treatment")
plt.ylabel("partial R² with outcome")
plt.xlim(0, 0.1)
plt.ylim(0, 0.7)
plt.legend(frameon=False)
plt.show()
print(f"relationship strength: partial R² {u_t:.3f} with treatment, {u_y:.3f} with outcome")

The real hidden confounder sits right next to the 3x-experience benchmark, and the bias-adjusted estimate for that benchmark lands close to the truth. The sensitivity analysis didn't know the answer, but it bracketed it.

Two more numbers to keep in mind for the triangulation:

In [ ]:
print(f"true ATT, North reps:    {truth['att_north_rep_abs']:.2f} k$ = {truth['att_north_rep_pct']:.1%}")
print(f"true ATE, all 240 reps:  {truth['ate_rep_abs']:.2f} k$ = {truth['ate_rep_pct']:.1%}")

The ATE is bigger than the ATT: the North reps are more experienced, and experienced reps get less out of the recommender. Rolling it out everywhere would pay off more than the North suggests.